# 02 — Training, optimisation and evaluation

## 1. Purpose

**What:** train, optimise and evaluate the sentiment models (negative / neutral / positive)
on the cleaned Financial PhraseBank data. This notebook is extended step by step: each
modelling iteration gets its own section and one row in `outputs/metrics/iterations.csv`.

**Why:** a fixed, documented protocol makes the iterations comparable and the final result
trustworthy.

> **Integrity rule:** the 20 % test set stays untouched until the final evaluation.
> All model choices (features, hyperparameters, model type) are made with cross-validation
> on the training part only. The test set is evaluated exactly once, for the final model.

In [1]:
# Imports and shared settings
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

from newsentiment import config
from newsentiment.data import load_training_data
from newsentiment.evaluation import (
    create_split,
    cross_val_predictions,
    cross_validate_model,
    evaluate,
    load_split,
    log_iteration,
    tune_model,
)
from newsentiment.model import build_pipeline, key_params

LABELS = list(config.LABELS)  # fixed class order: negative, neutral, positive

## 2. Data and split

**What:** load the cleaned data, create the stratified 80/20 split and apply it.
`create_split()` writes `data/split.json` (row positions plus a fingerprint of the data) on the
first run and simply reuses the file afterwards; `load_split()` refuses to apply it if the data
has changed.

**Why:** a stored split guarantees that every iteration, and the final evaluation, sees exactly
the same training and test rows. Stratification keeps the class shares equal in both parts, which
matters because the classes are imbalanced.

In [2]:
# Cleaned training data (see 01_eda.ipynb for the cleaning steps)
df, report = load_training_data()

# Create the split on the first run, reuse data/split.json afterwards
split = create_split(df)
train_df, test_df = load_split(df)

print(f"Rows: {len(df):,} | split file: {config.SPLIT_FILE.relative_to(config.PROJECT_ROOT).as_posix()}")

Rows: 4,836 | split file: data/split.json


In [3]:
# Size and class counts per part; the shares should be (almost) identical
parts = {"full": df, "train": train_df, "test": test_df}
split_table = pd.DataFrame(
    {name: part["label"].value_counts().reindex(LABELS) for name, part in parts.items()}
).T
split_table["total"] = split_table.sum(axis=1)
for label in LABELS:
    split_table[f"{label}_%"] = (split_table[label] / split_table["total"] * 100).round(1)
split_table

label,negative,neutral,positive,total,negative_%,neutral_%,positive_%
full,604,2871,1361,4836,12.5,59.4,28.1
train,483,2296,1089,3868,12.5,59.4,28.2
test,121,575,272,968,12.5,59.4,28.1


## 3. Evaluation protocol

**What:** every candidate model is scored with **5-fold stratified cross-validation on the
training part** (shuffled, fixed seed, so all models see identical folds). We report mean ± standard
deviation over the folds.

**Main metric: macro-F1** — the unweighted mean of the per-class F1 scores. With about 59 %
neutral sentences, accuracy rewards a model that mostly predicts "neutral"; macro-F1 gives the
small negative class the same weight as the others. Accuracy is reported as a secondary metric.
For the final model, per-class precision/recall and the confusion matrix are added
(`evaluate()` in `newsentiment.evaluation`).

## 4. Iteration "baseline": majority class

**What:** a `DummyClassifier` that always predicts the most frequent class (neutral), ignoring
the text.

**Why:** it is the lower bound every real model must clearly beat. Its accuracy equals the
neutral share, while its macro-F1 is low (F1 = 0 for negative and positive), which shows why
accuracy alone would be misleading.

In [4]:
# Cross-validate the majority-class baseline on the training part only
baseline = DummyClassifier(strategy="most_frequent")
cv_baseline = cross_validate_model(baseline, train_df["text"], train_df["label"])

print(f"Accuracy: {cv_baseline['accuracy_mean']:.3f} ± {cv_baseline['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_baseline['macro_f1_mean']:.3f} ± {cv_baseline['macro_f1_std']:.3f}")

# Per-fold scores
pd.DataFrame(
    {"accuracy": cv_baseline["fold_accuracy"], "macro_f1": cv_baseline["fold_macro_f1"]},
    index=pd.RangeIndex(1, config.CV_FOLDS + 1, name="fold"),
).round(4)

Accuracy: 0.594 ± 0.000
Macro-F1: 0.248 ± 0.000


,accuracy,macro_f1
fold,,
1,0.5943,0.2485
2,0.5930,0.2482
3,0.5930,0.2482
4,0.5938,0.2484
5,0.5938,0.2484


In [5]:
# Log the iteration (re-running replaces the existing "baseline" row)
iterations = log_iteration(
    "baseline",
    "Majority class",
    "DummyClassifier(strategy='most_frequent'); lower bound for all later models",
    cv_baseline,
    baseline.get_params(),
)
iterations

,iteration,model,description,cv_accuracy_mean,cv_accuracy_std,cv_macro_f1_mean,cv_macro_f1_std,params,logged_at
0,baseline,Majority class,DummyClassifier(strategy='most_frequent'); low...,0.5936,0.0005,0.2483,0.0001,"{""constant"": null, ""random_state"": null, ""stra...",2026-11-10T15:05:44
1,v0,TF-IDF + Multinomial NB,"TF-IDF unigrams (defaults, no stop-word remova...",0.6657,0.0126,0.4116,0.0218,"{""classifier"": ""MultinomialNB"", ""clf__alpha"": ...",2026-11-10T15:03:32
2,v1a,TF-IDF + logistic regression,TF-IDF unigrams (defaults) + LogisticRegressio...,0.7482,0.0167,0.6443,0.0278,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T15:03:34
3,v1b,TF-IDF + logistic regression (balanced),"As v1a, with class_weight='balanced' to counte...",0.7549,0.0106,0.7100,0.0135,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T15:03:35
4,v2,"TF-IDF + logistic regression (balanced, tuned)","Grid search (60 candidates) over ngram_range, ...",0.7728,0.0053,0.7275,0.0058,"{""classifier"": ""LogisticRegression"", ""clf__C"":...",2026-11-10T15:04:37
5,v2_nb,TF-IDF + Multinomial NB (tuned),"Grid search (40 candidates) over ngram_range, ...",0.7378,0.0146,0.6785,0.0156,"{""classifier"": ""MultinomialNB"", ""clf__alpha"": ...",2026-11-10T15:04:50


## 5. Iteration v0: TF-IDF + Multinomial Naive Bayes

**What:** the first text model. `TfidfVectorizer` with default settings (unigrams, lowercase,
**no stop-word removal**) turns each sentence into a weighted bag of words; Multinomial Naive Bayes
(default `alpha=1.0`) classifies it.

**Why:** Naive Bayes is the classic fast baseline for text classification. It shows how much a
plain bag of words already gains over the majority baseline. Stop words are kept on purpose:
scikit-learn's list contains *up, down, less, more, not, no*, which carry sentiment in financial news.
The vectorizer is part of the pipeline, so it is refitted inside every fold and never sees the
held-out sentences.

In [6]:
# Training inputs and labels (the test part is not used here)
X_train, y_train = train_df["text"], train_df["label"]

# v0: TF-IDF (unigrams, defaults) + Multinomial Naive Bayes
v0 = build_pipeline("nb")
cv_v0 = cross_validate_model(v0, X_train, y_train)

print(f"Accuracy: {cv_v0['accuracy_mean']:.3f} ± {cv_v0['accuracy_std']:.3f}")
print(f"Macro-F1: {cv_v0['macro_f1_mean']:.3f} ± {cv_v0['macro_f1_std']:.3f}")

# Log the iteration (re-running replaces the existing "v0" row)
log_iteration(
    "v0",
    "TF-IDF + Multinomial NB",
    "TF-IDF unigrams (defaults, no stop-word removal) + MultinomialNB(alpha=1.0)",
    cv_v0,
    key_params(v0),
);

Accuracy: 0.666 ± 0.013
Macro-F1: 0.412 ± 0.022


## 6. Iteration v1: TF-IDF + logistic regression

**What:** the same TF-IDF features with logistic regression, the main model of the concept.
- **v1a:** no class weights (every sentence counts equally).
- **v1b:** `class_weight="balanced"`: errors on a class are weighted inversely to its frequency,
  so the small negative class counts as much as the large neutral class.

**Why:** logistic regression learns the word weights discriminatively and, unlike Naive Bayes,
does not assume that words are independent. Class weights address the imbalance (about 12 %
negative sentences) directly in the loss; comparing v1a and v1b isolates their effect.

In [7]:
# v1a: logistic regression without class weights
v1a = build_pipeline("logreg")
cv_v1a = cross_validate_model(v1a, X_train, y_train)
log_iteration(
    "v1a",
    "TF-IDF + logistic regression",
    "TF-IDF unigrams (defaults) + LogisticRegression(C=1.0), no class weights",
    cv_v1a,
    key_params(v1a),
)

# v1b: same model, errors weighted inversely to class frequency
v1b = build_pipeline("logreg", clf__class_weight="balanced")
cv_v1b = cross_validate_model(v1b, X_train, y_train)
log_iteration(
    "v1b",
    "TF-IDF + logistic regression (balanced)",
    "As v1a, with class_weight='balanced' to counter the class imbalance",
    cv_v1b,
    key_params(v1b),
)

for name, cv in [("v1a", cv_v1a), ("v1b", cv_v1b)]:
    print(
        f"{name}: accuracy {cv['accuracy_mean']:.3f} ± {cv['accuracy_std']:.3f} | "
        f"macro-F1 {cv['macro_f1_mean']:.3f} ± {cv['macro_f1_std']:.3f}"
    )

v1a: accuracy 0.748 ± 0.017 | macro-F1 0.644 ± 0.028
v1b: accuracy 0.755 ± 0.011 | macro-F1 0.710 ± 0.013


## 7. Comparison

**What:** (a) the cross-validation scores of all iterations so far, read back from
`iterations.csv`; (b) per-class precision, recall and F1 from **out-of-fold predictions**:
`cross_val_predictions()` uses the same five folds, and every prediction comes from a model that
did not see that sentence. The per-class table is saved as `outputs/metrics/cv_per_class.csv`.

**Why:** the mean macro-F1 decides which model is best; the per-class view shows *where* the
models differ, especially on the small negative class. Both come from the training part only.

In [8]:
# (a) CV scores of all iterations, as logged
log = pd.read_csv(config.ITERATIONS_FILE)
comparison = pd.DataFrame(
    {
        "model": log["model"],
        "CV accuracy": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_accuracy_mean"], log["cv_accuracy_std"])],
        "CV macro-F1": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_macro_f1_mean"], log["cv_macro_f1_std"])],
    }
)
comparison.index = log["iteration"]
comparison

,model,CV accuracy,CV macro-F1
iteration,,,
baseline,Majority class,0.594 ± 0.001,0.248 ± 0.000
v0,TF-IDF + Multinomial NB,0.666 ± 0.013,0.412 ± 0.022
v1a,TF-IDF + logistic regression,0.748 ± 0.017,0.644 ± 0.028
v1b,TF-IDF + logistic regression (balanced),0.755 ± 0.011,0.710 ± 0.013
v2,"TF-IDF + logistic regression (balanced, tuned)",0.773 ± 0.005,0.728 ± 0.006
v2_nb,TF-IDF + Multinomial NB (tuned),0.738 ± 0.015,0.678 ± 0.016


In [9]:
# (b) Per-class metrics from out-of-fold predictions (same folds as above)
models = {"v0": v0, "v1a": v1a, "v1b": v1b}
rows = []
for name, model in models.items():
    oof_pred = cross_val_predictions(model, X_train, y_train)
    result = evaluate(y_train, oof_pred)
    for label in LABELS:
        rows.append({"iteration": name, "class": label, **result["per_class"][label]})

per_class = pd.DataFrame(rows, columns=["iteration", "class", "precision", "recall", "f1", "support"])
per_class_file = config.METRICS_DIR / "cv_per_class.csv"
per_class.round(4).to_csv(per_class_file, index=False)

# Wide view: one row per iteration, metrics per class side by side
per_class.pivot(index="iteration", columns="class", values=["precision", "recall", "f1"]).round(3)

precision                    recall                        f1  \
class      negative neutral positive negative neutral positive negative   
iteration                                                                 
v0            0.929   0.671    0.623    0.027   0.983    0.281    0.052   
v1a           0.823   0.743    0.747    0.366   0.949    0.493    0.507   
v1b           0.641   0.825    0.659    0.654   0.822    0.657    0.648   

                            
class     neutral positive  
iteration                   
v0          0.797    0.387  
v1a         0.834    0.594  
v1b         0.824    0.658

## 8. Observations

Measured with 5-fold cross-validation on the training part (3,868 sentences); the test set was not used.

- **All three text models clearly beat the majority baseline** (macro-F1 0.248):
  v0 (Naive Bayes) 0.412 ± 0.022, v1a (logistic regression) 0.644 ± 0.028,
  v1b (logistic regression, balanced) 0.710 ± 0.013.
- **Best model by macro-F1: v1b** (0.710), also the highest accuracy (0.755 ± 0.011) and the
  smallest fold-to-fold spread of the three text models.
- **Naive Bayes with defaults almost never predicts "negative":** out-of-fold recall 0.027
  (13 of 483 negative sentences), with neutral recall 0.983. Its accuracy (0.666) is only
  7 points above the baseline.
- **Effect of class weights on negative recall:** 0.366 (v1a) → 0.654 (v1b); negative F1
  0.507 → 0.648. Positive recall rises as well (0.493 → 0.657). The cost is lower negative
  precision (0.823 → 0.641) and lower neutral recall (0.949 → 0.822).

v1b is the starting point for the next iterations (n-grams and hyperparameter tuning).

## 9. Iteration v2: tuned logistic regression

**What:** a grid search over the TF-IDF settings and the regularisation strength of the best model
so far (v1b, logistic regression with `class_weight="balanced"`, which stays fixed):

| parameter | values | meaning |
|---|---|---|
| `tfidf__ngram_range` | (1, 1), (1, 2) | unigrams only, or unigrams + bigrams ("not increase", "fell to") |
| `tfidf__min_df` | 1, 2, 3 | drop words seen in fewer than *n* training sentences (rare words, typos, numbers) |
| `tfidf__sublinear_tf` | False, True | use 1 + log(tf) instead of the raw term count |
| `clf__C` | 0.1, 0.3, 1, 3, 10 | inverse regularisation strength (small C = stronger regularisation) |

2 × 3 × 2 × 5 = **60 candidates**, each scored on the same 5 folds with macro-F1 as before
(`tune_model()` wraps `GridSearchCV`). All results are saved to `outputs/metrics/grid_v2_logreg.csv`.

**Why:** v1b used default settings. A systematic search shows which settings actually help and
by how much. v1b's settings are part of the grid, so v2 cannot score lower than v1b; a lower
score would point to a bug. Picking the best of 60 CV scores is slightly optimistic; the
unbiased estimate comes from the test set at the very end.

In [10]:
# v2: grid search for logistic regression (class weights fixed to "balanced")
grid_v2 = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2, 3],
    "tfidf__sublinear_tf": [False, True],
    "clf__C": [0.1, 0.3, 1, 3, 10],
}
best_v2, results_v2 = tune_model(
    build_pipeline("logreg", clf__class_weight="balanced"), grid_v2, X_train, y_train
)

# Save every candidate; show the ten best
results_v2.to_csv(config.METRICS_DIR / "grid_v2_logreg.csv", index=False)
print(f"Candidates: {len(results_v2)} | best parameters: {best_v2}")
results_v2.head(10).round(4)

Candidates: 60 | best parameters: {'clf__C': 3, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


,clf__C,tfidf__min_df,tfidf__ngram_range,tfidf__sublinear_tf,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,rank
0,3.0,2,"(1, 2)",True,0.7275,0.0058,0.7728,0.0053,1
1,10.0,2,"(1, 2)",True,0.7274,0.0071,0.7751,0.0064,2
2,10.0,2,"(1, 2)",False,0.7253,0.0091,0.7725,0.0082,3
3,3.0,3,"(1, 2)",False,0.7251,0.0095,0.7678,0.0091,4
4,3.0,2,"(1, 2)",False,0.7247,0.0057,0.7691,0.0075,5
5,3.0,3,"(1, 2)",True,0.7244,0.0055,0.7673,0.0079,6
6,10.0,1,"(1, 1)",True,0.7226,0.0109,0.7668,0.0106,7
7,10.0,3,"(1, 2)",True,0.7226,0.0128,0.7678,0.0109,8
8,10.0,1,"(1, 2)",True,0.7213,0.0086,0.7735,0.0063,9
9,10.0,1,"(1, 2)",False,0.7212,0.0097,0.7727,0.0091,10


In [11]:
# Re-check: the best pipeline, cross-validated on its own, must reproduce the grid's best score
v2 = build_pipeline("logreg", clf__class_weight="balanced", **best_v2)
cv_v2 = cross_validate_model(v2, X_train, y_train)
assert abs(cv_v2["macro_f1_mean"] - results_v2.loc[0, "mean_macro_f1"]) < 1e-9, "re-check failed"
# v1b's settings are inside the grid, so v2 must be at least as good
assert cv_v2["macro_f1_mean"] >= cv_v1b["macro_f1_mean"] - 1e-9, "v2 worse than v1b"

print(f"Accuracy: {cv_v2['accuracy_mean']:.4f} ± {cv_v2['accuracy_std']:.4f}")
print(f"Macro-F1: {cv_v2['macro_f1_mean']:.4f} ± {cv_v2['macro_f1_std']:.4f}")

# Log the iteration (re-running replaces the existing "v2" row)
log_iteration(
    "v2",
    "TF-IDF + logistic regression (balanced, tuned)",
    "Grid search (60 candidates) over ngram_range, min_df, sublinear_tf and C; class_weight='balanced'",
    cv_v2,
    key_params(v2),
);

Accuracy: 0.7728 ± 0.0053
Macro-F1: 0.7275 ± 0.0058


## 10. Iteration v2_nb: tuned Naive Bayes

**What:** the same kind of grid search for Multinomial Naive Bayes:

| parameter | values | meaning |
|---|---|---|
| `tfidf__ngram_range` | (1, 1), (1, 2) | unigrams, or unigrams + bigrams |
| `tfidf__min_df` | 1, 2 | minimum number of sentences a word must appear in |
| `clf__alpha` | 0.01, 0.03, 0.1, 0.3, 1.0 | additive smoothing (v0 used 1.0) |
| `clf__fit_prior` | True, False | learn class priors from the data, or use uniform priors |

2 × 2 × 5 × 2 = **40 candidates**, saved to `outputs/metrics/grid_v2_nb.csv`.

**Why:** v0 used Naive Bayes with defaults, while logistic regression is now tuned; comparing a
tuned model with an untuned one would be unfair. `fit_prior=False` is Naive Bayes' counterpart to
class weights: it removes the pull towards the frequent neutral class.

In [12]:
# v2_nb: grid search for Multinomial Naive Bayes
grid_v2_nb = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__alpha": [0.01, 0.03, 0.1, 0.3, 1.0],
    "clf__fit_prior": [True, False],
}
best_v2_nb, results_v2_nb = tune_model(build_pipeline("nb"), grid_v2_nb, X_train, y_train)

# Save every candidate; show the ten best
results_v2_nb.to_csv(config.METRICS_DIR / "grid_v2_nb.csv", index=False)
print(f"Candidates: {len(results_v2_nb)} | best parameters: {best_v2_nb}")
results_v2_nb.head(10).round(4)

Candidates: 40 | best parameters: {'clf__alpha': 0.3, 'clf__fit_prior': False, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}


,clf__alpha,clf__fit_prior,tfidf__min_df,tfidf__ngram_range,mean_macro_f1,std_macro_f1,mean_accuracy,std_accuracy,rank
0,0.30,False,2,"(1, 2)",0.6785,0.0156,0.7378,0.0146,1
1,0.10,False,2,"(1, 2)",0.6694,0.0130,0.7231,0.0131,2
2,0.10,False,1,"(1, 2)",0.6658,0.0196,0.7316,0.0188,3
3,0.10,True,2,"(1, 2)",0.6658,0.0212,0.7440,0.0186,4
4,0.03,True,2,"(1, 2)",0.6653,0.0264,0.7394,0.0216,5
5,0.03,True,1,"(1, 2)",0.6627,0.0117,0.7358,0.0120,6
6,0.03,False,2,"(1, 2)",0.6596,0.0127,0.7182,0.0135,7
7,0.01,True,2,"(1, 2)",0.6576,0.0200,0.7347,0.0177,8
8,0.01,True,1,"(1, 2)",0.6576,0.0109,0.7278,0.0095,9
9,0.30,False,1,"(1, 1)",0.6555,0.0223,0.7151,0.0192,10


In [13]:
# Re-check: the best Naive Bayes pipeline must reproduce the grid's best score
v2_nb = build_pipeline("nb", **best_v2_nb)
cv_v2_nb = cross_validate_model(v2_nb, X_train, y_train)
assert abs(cv_v2_nb["macro_f1_mean"] - results_v2_nb.loc[0, "mean_macro_f1"]) < 1e-9, "re-check failed"

print(f"Accuracy: {cv_v2_nb['accuracy_mean']:.4f} ± {cv_v2_nb['accuracy_std']:.4f}")
print(f"Macro-F1: {cv_v2_nb['macro_f1_mean']:.4f} ± {cv_v2_nb['macro_f1_std']:.4f}")

# Log the iteration (re-running replaces the existing "v2_nb" row)
log_iteration(
    "v2_nb",
    "TF-IDF + Multinomial NB (tuned)",
    "Grid search (40 candidates) over ngram_range, min_df, alpha and fit_prior",
    cv_v2_nb,
    key_params(v2_nb),
);

Accuracy: 0.7378 ± 0.0146
Macro-F1: 0.6785 ± 0.0156


## 11. Comparison after tuning

**What:** (a) the CV scores of all iterations from `iterations.csv`; (b) out-of-fold per-class
precision, recall and F1 for v2 and v2_nb, added to `outputs/metrics/cv_per_class.csv` (rows of
the same iteration are replaced, so re-runs create no duplicates); (c) the macro-F1 gains,
compared with the fold-to-fold standard deviation and fold by fold (all models use the same folds).

**Why:** a gain from tuning is only meaningful if it is large relative to the noise between
folds; the per-class view shows whether tuning changed the weak negative class.

In [14]:
# (a) CV scores of all iterations, as logged
log = pd.read_csv(config.ITERATIONS_FILE)
comparison = pd.DataFrame(
    {
        "model": log["model"],
        "CV accuracy": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_accuracy_mean"], log["cv_accuracy_std"])],
        "CV macro-F1": [f"{m:.3f} ± {s:.3f}" for m, s in zip(log["cv_macro_f1_mean"], log["cv_macro_f1_std"])],
    }
)
comparison.index = log["iteration"]
comparison

,model,CV accuracy,CV macro-F1
iteration,,,
baseline,Majority class,0.594 ± 0.001,0.248 ± 0.000
v0,TF-IDF + Multinomial NB,0.666 ± 0.013,0.412 ± 0.022
v1a,TF-IDF + logistic regression,0.748 ± 0.017,0.644 ± 0.028
v1b,TF-IDF + logistic regression (balanced),0.755 ± 0.011,0.710 ± 0.013
v2,"TF-IDF + logistic regression (balanced, tuned)",0.773 ± 0.005,0.728 ± 0.006
v2_nb,TF-IDF + Multinomial NB (tuned),0.738 ± 0.015,0.678 ± 0.016


In [15]:
# (b) Out-of-fold per-class metrics for the tuned models (same folds as above)
new_rows = []
for name, model in {"v2": v2, "v2_nb": v2_nb}.items():
    result = evaluate(y_train, cross_val_predictions(model, X_train, y_train))
    for label in LABELS:
        new_rows.append({"iteration": name, "class": label, **result["per_class"][label]})
new_per_class = pd.DataFrame(new_rows).round(4)

# Upsert: drop old rows of these iterations, then append the new ones
per_class_file = config.METRICS_DIR / "cv_per_class.csv"
per_class_all = pd.read_csv(per_class_file)
per_class_all = per_class_all[~per_class_all["iteration"].isin(new_per_class["iteration"])]
per_class_all = pd.concat([per_class_all, new_per_class], ignore_index=True)
per_class_all.to_csv(per_class_file, index=False)

# Wide view, iterations in log order
order = [i for i in log["iteration"] if i in set(per_class_all["iteration"])]
(
    per_class_all.pivot(index="iteration", columns="class", values=["precision", "recall", "f1"])
    .reindex(order)
    .round(3)
)

precision                    recall                        f1  \
class      negative neutral positive negative neutral positive negative   
iteration                                                                 
v0            0.929   0.671    0.623    0.027   0.983    0.281    0.052   
v1a           0.823   0.743    0.747    0.366   0.950    0.493    0.507   
v1b           0.641   0.825    0.659    0.654   0.822    0.658    0.648   
v2            0.688   0.826    0.689    0.652   0.851    0.661    0.670   
v2_nb         0.609   0.804    0.641    0.586   0.836    0.598    0.597   

                            
class     neutral positive  
iteration                   
v0          0.797    0.387  
v1a         0.834    0.594  
v1b         0.824    0.658  
v2          0.838    0.675  
v2_nb       0.820    0.619

In [16]:
# (c) Macro-F1 gains relative to the fold standard deviation; fold-wise differences are
# paired (same folds), so they show how consistent a gain is
pairs = {"v2 vs v1b": (cv_v2, cv_v1b), "v2_nb vs v0": (cv_v2_nb, cv_v0), "v2 vs v2_nb": (cv_v2, cv_v2_nb)}
gain_rows = []
for name, (a, b) in pairs.items():
    diff = np.subtract(a["fold_macro_f1"], b["fold_macro_f1"])
    gain_rows.append(
        {
            "comparison": name,
            "macro_f1_gain": a["macro_f1_mean"] - b["macro_f1_mean"],
            "fold_std_first": a["macro_f1_std"],
            "fold_std_second": b["macro_f1_std"],
            "folds_won": f"{int((diff > 0).sum())}/{len(diff)}",
            "fold_diff_min": diff.min(),
            "fold_diff_max": diff.max(),
        }
    )
pd.DataFrame(gain_rows).set_index("comparison").round(4)

,macro_f1_gain,fold_std_first,fold_std_second,folds_won,fold_diff_min,fold_diff_max
comparison,,,,,,
v2 vs v1b,0.0175,0.0058,0.0135,5/5,0.0041,0.0282
v2_nb vs v0,0.2669,0.0156,0.0218,5/5,0.2185,0.3117
v2 vs v2_nb,0.0490,0.0058,0.0156,5/5,0.0284,0.0739


## 12. Observations

Measured with 5-fold cross-validation on the training part (3,868 sentences), the same folds as
all earlier iterations; the test set was not used.

- **Best logistic regression (v2):** `ngram_range=(1, 2)`, `min_df=2`, `sublinear_tf=True`, `C=3`
  (with `class_weight="balanced"`). CV macro-F1 **0.7275 ± 0.0058**, accuracy 0.7728 ± 0.0053.
  v1b's settings rank 33rd of 60 in this grid, with exactly v1b's logged scores. The runner-up
  (`C=10`, otherwise identical) scores 0.7274, only 0.0001 lower.
- **Best Naive Bayes (v2_nb):** `ngram_range=(1, 2)`, `min_df=2`, `alpha=0.3`, `fit_prior=False`.
  CV macro-F1 **0.6785 ± 0.0156**, accuracy 0.7378 ± 0.0146. Tuning raises Naive Bayes by 0.267
  macro-F1 over v0 (0.412), better on all 5 folds. All ten best NB candidates use bigrams.
- **Gain of v2 over v1b:** +0.0175 macro-F1 (0.7100 → 0.7275). That is larger than the fold
  standard deviations of both models (0.0058 and 0.0135), and v2 is better on all 5 folds
  (fold-wise gain 0.004 to 0.028). Part of this gain is selection optimism (best of 60
  candidates); the test set will give the unbiased figure. The fold spread also shrank
  (0.0135 → 0.0058).
- **v2 vs v2_nb:** logistic regression stays ahead by 0.049 macro-F1 (0.7275 vs 0.6785) and
  0.035 accuracy, on all 5 folds (fold-wise gain 0.028 to 0.074). The fair, tuned comparison
  confirms logistic regression as the main model.
- **Negative recall of v2:** 0.652 (315 of 483 negative sentences), practically the same as
  v1b (0.654, 316 of 483). Tuning improved negative *precision* (0.641 → 0.688, F1 0.648 → 0.670)
  and neutral recall (0.822 → 0.851), not negative recall. About a third of negative sentences
  are still missed. v2_nb's negative recall is lower: 0.586.

v2 is the current best model and the candidate for the final evaluation.

## 13. Error analysis of v2: out-of-fold predictions and probabilities

**What:** sections 13–18 analyse *where* and *why* v2 fails. v2 is rebuilt from the parameters
logged in `iterations.csv` (not from a variable in memory), and `cross_val_predictions()` /
`cross_val_probabilities()` give one out-of-fold prediction and one probability row per training
sentence, from the same five folds as all iterations. The confusion matrix is shown as counts
and as row percentages (rows = true class, so each row shows how that class was predicted); the
off-diagonal cells are then sorted by count.

**Why:** out-of-fold predictions come from models that never saw the sentence, so the errors are
honest without touching the test set. This analysis changes no model and logs no iteration; it
prepares the choice of one targeted improvement (v3).

In [17]:
import json
import re

from newsentiment.evaluation import cross_val_probabilities

# Rebuild v2 from the logged parameters (single source of truth: iterations.csv)
log = pd.read_csv(config.ITERATIONS_FILE)
v2_logged = json.loads(log.loc[log["iteration"] == "v2", "params"].iloc[0])
assert v2_logged.pop("classifier") == "LogisticRegression"
# JSON stores tuples as lists, e.g. ngram_range [1, 2] -> (1, 2)
v2_params = {k: tuple(v) if isinstance(v, list) else v for k, v in v2_logged.items()}
v2_rebuilt = build_pipeline("logreg", **v2_params)
print("v2 parameters:", v2_params)

# Out-of-fold predictions and probabilities (columns in LABELS order)
oof_pred = cross_val_predictions(v2_rebuilt, X_train, y_train)
oof_proba = cross_val_probabilities(v2_rebuilt, X_train, y_train)
assert list(v2_rebuilt.fit(X_train, y_train).classes_) == LABELS  # column order check
# The predicted class is the most probable one
assert (np.array(LABELS)[oof_proba.argmax(axis=1)] == oof_pred).all()

# Consistency with section 11: same per-class recall as in cv_per_class.csv
result_v2 = evaluate(y_train, oof_pred)
cm = np.array(result_v2["confusion_matrix"])
assert cm.sum() == len(train_df) == 3868
logged_recall = pd.read_csv(config.METRICS_DIR / "cv_per_class.csv").query("iteration == 'v2'")
for label, recall in zip(logged_recall["class"], logged_recall["recall"]):
    assert round(result_v2["per_class"][label]["recall"], 4) == recall, label
print(f"Out-of-fold macro-F1: {result_v2['macro_f1']:.4f} | accuracy: {result_v2['accuracy']:.4f}")

v2 parameters: {'clf__C': 3, 'clf__class_weight': 'balanced', 'tfidf__max_df': 1.0, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}


Out-of-fold macro-F1: 0.7276 | accuracy: 0.7728


In [18]:
# Confusion matrix as counts and as row percentages (rows = true class)
cm_counts = pd.DataFrame(
    cm,
    index=pd.Index(LABELS, name="true"),
    columns=pd.Index(LABELS, name="predicted"),
)
cm_row_pct = (cm_counts.div(cm_counts.sum(axis=1), axis=0) * 100).round(1)
display(cm_counts)
display(cm_row_pct)

predicted,negative,neutral,positive
true,,,
negative,315,100,68
neutral,85,1954,257
positive,58,311,720


predicted,negative,neutral,positive
true,,,
negative,65.2,20.7,14.1
neutral,3.7,85.1,11.2
positive,5.3,28.6,66.1


In [19]:
# Off-diagonal cells (the error types), largest first
off_diag = pd.DataFrame(
    [
        {
            "true": t,
            "predicted": p,
            "count": int(cm_counts.loc[t, p]),
            "share_of_true_class_%": cm_row_pct.loc[t, p],
            "share_of_all_errors_%": round(cm_counts.loc[t, p] / (cm.sum() - np.trace(cm)) * 100, 1),
        }
        for t in LABELS
        for p in LABELS
        if t != p
    ]
).sort_values("count", ascending=False, kind="stable").reset_index(drop=True)
off_diag

,true,predicted,count,share_of_true_class_%,share_of_all_errors_%
0,positive,neutral,311,28.6,35.4
1,neutral,positive,257,11.2,29.2
2,negative,neutral,100,20.7,11.4
3,neutral,negative,85,3.7,9.7
4,negative,positive,68,14.1,7.7
5,positive,negative,58,5.3,6.6


## 14. What v2 learned: the strongest n-grams per class

**What:** v2 is fitted once on the whole training part (training data only, no test rows). For
each class, logistic regression has one coefficient per n-gram; a large positive coefficient
pushes a sentence towards that class. The 15 n-grams with the largest coefficients per class are
shown and saved to `outputs/metrics/v2_top_features.csv` (class, rank, ngram, coefficient).

**Why:** the coefficients show which cues the model relies on. If they are mostly generic
financial words, or tied to particular companies, that explains part of the errors.

In [20]:
# Fit v2 on the whole training part (no test rows)
v2_full = build_pipeline("logreg", **v2_params).fit(X_train, y_train)
ngrams = v2_full.named_steps["tfidf"].get_feature_names_out()
coefs = v2_full.named_steps["clf"].coef_  # shape (n_classes, n_features), rows in classes_ order
assert list(v2_full.named_steps["clf"].classes_) == LABELS

TOP_K = 15
top_rows = []
for c, label in enumerate(LABELS):
    for rank, j in enumerate(np.argsort(coefs[c])[::-1][:TOP_K], start=1):
        top_rows.append({"class": label, "rank": rank, "ngram": ngrams[j], "coefficient": round(float(coefs[c, j]), 4)})
top_features = pd.DataFrame(top_rows)
top_features.to_csv(config.METRICS_DIR / "v2_top_features.csv", index=False)

print(f"Vocabulary size: {len(ngrams):,} n-grams")
# Side-by-side view: one column pair (n-gram, coefficient) per class
top_features.pivot(index="rank", columns="class", values=["ngram", "coefficient"]).swaplevel(axis=1)[
    [(label, col) for label in LABELS for col in ("ngram", "coefficient")]
]

Vocabulary size: 12,574 n-grams


class      negative                    neutral                  positive  \
              ngram coefficient          ngram coefficient         ngram   
rank                                                                       
1              down       5.935             is      2.8491     increased   
2         decreased      5.2862  approximately      2.1078          rose   
3              fell      3.9229             no      1.8012      increase   
4             lower      2.8116    development      1.7963       up from   
5             staff      2.7738       business      1.7858            up   
6            result      2.6558       includes      1.7702          grew   
7         down from      2.5507          stake      1.5583      positive   
8           dropped      2.5448          range      1.4334        signed   
9          negative      2.4756          other      1.3215       rose to   
10     decreased to      2.4571            not       1.302       leading   
11              lay      2.3663       acquired       1.297        growth   
12              off       2.335       sales of        1.23            to   
13           to eur      2.1603      net sales      1.1992       awarded   
14             fall      2.1505        include      1.1688      improved   
15              had      2.0618       value of      1.1624  increased by   

class              
      coefficient  
rank               
1          3.9866  
2           3.933  
3          3.7971  
4          3.3147  
5          3.2025  
6          2.9517  
7          2.9193  
8           2.795  
9          2.7638  
10         2.6535  
11         2.6507  
12          2.617  
13         2.3629  
14         2.2415  
15         2.1879

## 15. Error list

**What:** every misclassified training sentence with its row id (position in the training part,
`train_df` index), text, true and predicted label, the probability of the predicted class and
the probability of the true class. The full list is saved to `outputs/analysis/v2_oof_errors.csv`.
Below, 8 random examples (seed `config.RANDOM_SEED`) are shown for each of the 4 largest
off-diagonal cells.

**Why:** reading concrete sentences shows patterns that summary metrics cannot (e.g. comparisons
between periods, figures without a sentiment word). The folder `outputs/analysis/` is
git-ignored because its files contain dataset sentences, which this project does not
redistribute.

In [21]:
# One row per training sentence with its out-of-fold result
label_pos = {label: i for i, label in enumerate(LABELS)}
rows_all = pd.DataFrame(
    {
        "row_id": train_df.index,
        "text": X_train.values,
        "true": y_train.values,
        "predicted": oof_pred,
        "p_predicted": oof_proba.max(axis=1),
        "p_true": oof_proba[np.arange(len(y_train)), y_train.map(label_pos).values],
    }
)
rows_all["correct"] = rows_all["true"] == rows_all["predicted"]

# Error list (git-ignored folder)
errors = rows_all.loc[~rows_all["correct"]].drop(columns="correct").reset_index(drop=True)
config.ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
errors_file = config.ANALYSIS_DIR / "v2_oof_errors.csv"
errors.round(4).to_csv(errors_file, index=False)
print(f"Misclassified: {len(errors):,} of {len(rows_all):,} -> {errors_file.relative_to(config.PROJECT_ROOT).as_posix()}")

Misclassified: 879 of 3,868 -> outputs/analysis/v2_oof_errors.csv


In [22]:
# 8 random examples for each of the 4 largest error cells
with pd.option_context("display.max_colwidth", None):
    for _, cell in off_diag.head(4).iterrows():
        subset = errors[(errors["true"] == cell["true"]) & (errors["predicted"] == cell["predicted"])]
        sample = subset.sample(n=min(8, len(subset)), random_state=config.RANDOM_SEED)
        print(f"true {cell['true']} -> predicted {cell['predicted']} ({cell['count']} sentences)")
        display(sample[["row_id", "text", "p_predicted", "p_true"]].round(3).set_index("row_id"))

true positive -> predicted neutral (311 sentences)


,text,p_predicted,p_true
row_id,,,
2794,"According to Finnish Aktia Bank 's Managing Director Jussi Laitinen , the bank 's aim is to triple the number of its customers in Finland in the next five years .",0.487,0.402
53,"The OMX Helsinki 25 index was up 0.92 pct at 2,518.67 and the Helsinki CAP portfolio index was 0.91 pct higher at 4,711.19 .",0.442,0.218
594,"A portion , $ 12.5 million , will be recorded as part of its winnings in a prior patent dispute with Finnish phone maker Nokia Oyj .",0.792,0.138
602,"During the past decade it has gradually divested noncore assets and bought several sports equipment makers , including California-based Fitness Products International and Sparks , Nevada-based ATEC , a leading maker of baseball and softball pitching machines .",0.520,0.388
255,YIT says the acquisition is a part of its strategy for expansion in Central and Eastern European markets .,0.546,0.439
612,"Of the price , Kesko 's share is 10 mln euro $ 15.5 mln and it will recognize a gain of 4.0 mln euro $ 6.2 mln on the disposal which will be included in the result for the second quarter of 2008 .",0.695,0.203
690,NORDIC BUSINESS REPORT-26 June 2006-Metso Corporation wins EUR50m equipment order in Australia -_ 1998-2006 M2 COMMUNICATIONS LTD The Finnish engineering and technology group Metso Corporation said on Monday ( 26 June ) that it has received a EUR50m equipment order in Australia .,0.485,0.465
1383,"Under a memorandum of understanding MoU the parties will collaborate in preparing and implementing joint projects in the fields of combined power and heat generation , environmental protection and energy efficient technologies .",0.718,0.206


true neutral -> predicted positive (257 sentences)


,text,p_predicted,p_true
row_id,,,
2376,"In terms of power generation , Fortum produced 39.3 TWh of electricity during the first nine months of 2006 , compared with 38.0 TWh during the same period in 2005 , according to figures released in the group 's latest financial results .",0.508,0.205
2611,"The fixed-term contract of Mr. Jarmo Ukonaho , the current General Manager of Incap 's Indian operations , will finish by the end of the year .",0.656,0.233
1262,More than 80 special events in the three counties during four months were hosted by the library system and its nine branch libraries .,0.407,0.399
454,"The company has also supplied more than 200 MW of power generating equipment for a number of projects in Papua New Guinea , including 12 engines of the Wartsila 32 type to Lihir Gold .",0.567,0.375
327,Pohjola could increase its stake to 45 % in 2013 .,0.542,0.421
321,The Finnish company Stockmann has signed the contract with the company Technopolis .,0.740,0.228
2780,"`` This agreement is a direct result of LCC 's strategy and plan that was announced earlier this year , '' said Dean Douglas , LCC 's chief executive officer .",0.512,0.120
406,"The acquisition is part of Suomen Helasto 's strategy to expand the LukkoExpert Security chain , Suomen Helasto CEO Kimmo Uusimaki said .",0.596,0.377


true negative -> predicted neutral (100 sentences)


,text,p_predicted,p_true
row_id,,,
3638,"HELSINKI Thomson Financial - Shares closed little changed , with Cargotec and Huhtamaki dropping sharply on disappointing second-quarter reports .",0.638,0.220
3473,The number of salaried employees and group administration employees to be dismissed from the Sievi units is 17 .,0.479,0.450
3535,The maritime administration said the ships had ignored warnings about the icy conditions .,0.595,0.234
3411,The production is to be liquidated before June 2009 and 325 employees loose their jobs .,0.557,0.371
3405,"The largest restructuring is planned for Italy , where negotiations on about 40 job cuts are to be initiated today , the company noted .",0.727,0.135
3363,"Initially , the company said a maximum of 15 people would be made redundant in the parent company and about 15 in its subsidiaries .",0.573,0.281
2910,"Konecranes has previously communicated an estimated reduction of about 1,600 employees on group level in 2009 .",0.390,0.302
3615,"Tallink Silja attributes the significant drop to problems with the booking system that was taken into operation in October , the sale of trailer ferry ` Sky Wind ' and the route between Stockholm and Riga , which has won passengers from the Helsinki-Stockholm route .",0.429,0.225


true neutral -> predicted negative (85 sentences)


,text,p_predicted,p_true
row_id,,,
3659,"Operating profit for the 12-month period decreased from EUR9 .6 m while net sales increased from EUR69 .0 m , as compared to 2005 .",0.873,0.028
55,"As a result of the share issue , the number of the company 's shares increases by 391,630 shares and the share capital by a total of EUR 15,665.20 .",0.512,0.386
3314,`` We have the most expensive water brand in Finland at the moment . '',0.739,0.157
1932,"As a result , a total of 82 employees would be given notice .",0.647,0.296
1246,It also confirmed its earnings guidance for the whole 2009 issued in its report for the whole 2008 .,0.760,0.165
3688,"- Among other Finnish shares traded in the US , Stora Enso closed 0.33 pct lower at 12.11 eur , UPM-Kymmene was up 0.28 pct at 18.10 eur , and Metso was down 0.13 pct at 31.06 eur .",0.858,0.054
1108,The EUR 0.7 million non-recurring expenses have been recorded for the third quarter .,0.642,0.106
1641,"The Board of Directors proposes to the Shareholders ' Meeting on 18 March 2010 that the company would pay dividend for the financial year January 1 - December 31 , 2009 , EUR 0.02 per share .",0.503,0.417


## 16. Diagnostics: correctly vs wrongly classified sentences

**What:** simple surface features, compared between correctly and wrongly classified sentences
(shares in %, means for length and confidence). All matches are case-insensitive and on whole
words or phrases:

| feature | rule |
|---|---|
| digit | any character 0–9 |
| period comparison | "from", "compared", "year-earlier", "corresponding period", "previous year" |
| "loss" | the word *loss* (not *losses*) |
| negation | "not", "no", "without", or the contraction "n't" |
| length | number of whitespace-separated words |
| confidence | probability of the predicted class (max probability) |

The table is saved to `outputs/metrics/v2_error_diagnostics.csv`.

**Why:** if a feature is much more common among errors, it marks a sentence type the bag-of-words
model handles badly, which is a candidate target for v3.

In [23]:
# Feature flags; \b keeps matches on whole words/phrases ("from" does not match "fromage")
PATTERNS = {
    "contains a digit (%)": r"\d",
    "contains a period comparison (%)": r"\b(?:from|compared|year-earlier|corresponding period|previous year)\b",
    "contains 'loss' (%)": r"\bloss\b",
    # "n't" is matched on its own: in the data it may be attached ("didn't") or split ("did n't")
    "contains a negation (%)": r"\b(?:not|no|without)\b|n't",
}
features = pd.DataFrame(
    {name: rows_all["text"].str.contains(pattern, flags=re.IGNORECASE, regex=True) for name, pattern in PATTERNS.items()}
)
features["mean length (words)"] = rows_all["text"].str.split().str.len()
features["mean confidence (max probability)"] = rows_all["p_predicted"]

groups = {"correct": rows_all["correct"], "wrong": ~rows_all["correct"]}
diagnostics = pd.DataFrame({name: features[mask].mean() for name, mask in groups.items()})
# Boolean flags -> percent
is_share = diagnostics.index.str.endswith("(%)")
diagnostics.loc[is_share] *= 100
diagnostics.loc["sentences (n)"] = [int(mask.sum()) for mask in groups.values()]
diagnostics = diagnostics.round(3)
diagnostics.index.name = "feature"
diagnostics.to_csv(config.METRICS_DIR / "v2_error_diagnostics.csv")
diagnostics

,correct,wrong
feature,,
contains a digit (%),55.269,47.440
contains a period comparison (%),18.167,12.856
contains 'loss' (%),2.944,2.730
contains a negation (%),3.981,2.503
mean length (words),23.136,23.328
mean confidence (max probability),0.717,0.584
sentences (n),2989.000,879.000


## 17. Accuracy by confidence

**What:** the sentences are grouped by the probability of the predicted class into the bins
[0, 0.5), [0.5, 0.7), [0.7, 0.9) and [0.9, 1.0]; per bin the number of sentences, their share
and the out-of-fold accuracy. Saved to `outputs/metrics/v2_confidence_bins.csv`.

**Why:** if accuracy rises with confidence, the probabilities are informative: low-confidence
sentences are where most errors sit, and confident errors point to systematic mistakes
(e.g. labels the model cannot learn from words alone).

In [24]:
# Bins of the max probability; the last bin includes 1.0
edges = [0.0, 0.5, 0.7, 0.9, np.inf]
bin_labels = ["[0, 0.5)", "[0.5, 0.7)", "[0.7, 0.9)", "[0.9, 1.0]"]
rows_all["confidence_bin"] = pd.cut(rows_all["p_predicted"], bins=edges, right=False, labels=bin_labels)

conf_bins = rows_all.groupby("confidence_bin", observed=False).agg(
    sentences=("correct", "size"),
    accuracy=("correct", "mean"),
    errors=("correct", lambda s: int((~s).sum())),
)
conf_bins.insert(1, "share_%", (conf_bins["sentences"] / len(rows_all) * 100).round(1))
conf_bins["accuracy"] = conf_bins["accuracy"].round(4)
assert conf_bins["sentences"].sum() == len(rows_all)
conf_bins.to_csv(config.METRICS_DIR / "v2_confidence_bins.csv")
conf_bins

,sentences,share_%,accuracy,errors
confidence_bin,,,,
"[0, 0.5)",524,13.5,0.5210,251
"[0.5, 0.7)",1484,38.4,0.6914,458
"[0.7, 0.9)",1510,39.0,0.8927,162
"[0.9, 1.0]",350,9.0,0.9771,8


## 18. Observations

Out-of-fold results of v2 on the training part (3,868 sentences, the same 5 folds as all
iterations); the test set was not used, no model was changed and no iteration was logged.

- **Totals:** 879 of 3,868 sentences are misclassified (accuracy 0.7728). Macro-F1 over the
  pooled out-of-fold predictions is 0.7276 (the logged 0.7275 is the mean of the five fold scores).
  Per-class recall matches `cv_per_class.csv`: negative 0.652, neutral 0.851, positive 0.661.
- **Neutral vs positive is the main confusion:** positive → neutral (311, 28.6 % of all positive
  sentences) and neutral → positive (257, 11.2 % of neutral) together are 568 errors, 64.6 % of
  all errors. Negative → neutral is third (100, 20.7 % of negative sentences), neutral → negative
  fourth (85). Negative and positive are rarely swapped (68 + 58 = 126, 14.3 % of errors).
- **Learned cues:** the strongest n-grams are direction words: negative *down, decreased, fell,
  lower, down from, dropped, decreased to, fall*; positive *increased, rose, increase, up from, up,
  grew, rose to, improved*. Some are topic words rather than sentiment words: *staff, lay, off*
  (negative: layoffs), *signed, awarded, leading* (positive: contracts, orders). The neutral class
  has smaller coefficients (largest 2.85 vs 5.94 negative, 3.99 positive), and *no* and *not*
  are among its 15 strongest n-grams. The vocabulary has 12,574 n-grams.
- **Examples (32 sentences shown in section 15):** many positive → neutral errors contain no
  direction word (orders, acquisitions, strategy statements, index figures). Several neutral →
  negative errors contain a direction word that does not carry the label (e.g. *decreased ...
  while net sales increased*, *closed 0.33 pct lower* in a market list). Negative → neutral
  errors often describe job cuts in words like *dismissed, redundant, job cuts, reduction of
  employees*.
- **Diagnostics (correct vs wrong):** a digit appears in 55.3 % vs 47.4 %; a period comparison
  in 18.2 % vs 12.9 %; *loss* in 2.9 % vs 2.7 %; a negation in 4.0 % vs 2.5 %. Mean length is
  almost equal (23.1 vs 23.3 words). None of these features is more frequent among the errors;
  digits and period comparisons are *less* frequent there. Mean confidence is 0.717 for correct
  and 0.584 for wrong predictions.
- **Confidence:** accuracy rises with the max probability: 0.521 below 0.5 (524 sentences,
  13.5 %), 0.691 in [0.5, 0.7) (1,484, 38.4 %), 0.893 in [0.7, 0.9) (1,510, 39.0 %), 0.977 in
  [0.9, 1.0] (350, 9.0 %). 709 of 879 errors (80.7 %) have a max probability below 0.7; only 8
  errors have a max probability of 0.9 or more.